# Watermark Removal

Solution Author: Asandei Stefan-Alexandru

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 64
LR = 2e-4
EPOCHS = 80
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/home/stefan/ioai-prep/kits/watermark-removal"
SUBMISSION_DIR = "./submission_rfm"

In [ ]:
SUBMISSION_DIR = "./submission_rfm"

# Data

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, clean_dir, wm_dir, transform=None):
        self.clean_dir = clean_dir
        self.wm_dir = wm_dir
        self.images = sorted(os.listdir(wm_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]
        wm = Image.open(os.path.join(self.wm_dir, name)).convert("RGB")
        clean = (
            Image.open(os.path.join(self.clean_dir, name)).convert("RGB")
            if self.clean_dir
            else wm
        )
        if self.transform:
            wm = self.transform(wm)
            clean = self.transform(clean)
        return wm, clean, name

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

dataset = BirdDataset(
    os.path.join(OUT_DIR, "train/clean"),
    os.path.join(OUT_DIR, "train/watermarked"),
    transform,
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Model

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_mlp = nn.Linear(t_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        )
        self.relu = nn.SiLU()

    def forward(self, x, t):
        h = self.relu(self.norm1(x))
        h = self.conv1(h)
        h += self.t_mlp(t)[:, :, None, None]
        h = self.relu(self.norm2(h))
        return self.conv2(h) + self.shortcut(x)

In [ ]:
class FlowUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.t_embed = nn.Sequential(
            nn.Linear(256, 512), nn.SiLU(), nn.Linear(512, 512)
        )
        self.init_conv = nn.Conv2d(6, 64, 3, padding=1)
        self.down1 = ResBlock(64, 128, 512)
        self.down2 = ResBlock(128, 256, 512)
        self.mid = ResBlock(256, 256, 512)
        self.up1 = ResBlock(256 + 128, 128, 512)
        self.up2 = ResBlock(128 + 64, 64, 512)
        self.out = nn.Conv2d(64, 3, 1)
        self.pool = nn.MaxPool2d(2)

    def pos_encoding(self, t, channels):
        # t is in [0, 1], scale to 1000 for standard PE frequency
        t = t * 1000
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, channels, 2).float() / channels)
        ).to(DEVICE)
        pos = t.unsqueeze(1) * inv_freq
        return torch.cat([pos.sin(), pos.cos()], dim=-1)

    def forward(self, x, t, cond):
        t_emb = self.t_embed(self.pos_encoding(t, 256))
        x = torch.cat([x, cond], dim=1)
        x1 = self.init_conv(x)
        x2 = self.down1(self.pool(x1), t_emb)
        x3 = self.down2(self.pool(x2), t_emb)
        x3 = self.mid(x3, t_emb)
        x_up = nn.functional.interpolate(x3, scale_factor=2, mode="bilinear")
        u1 = self.up1(torch.cat([x_up, x2], dim=1), t_emb)
        u1_up = nn.functional.interpolate(u1, scale_factor=2, mode="bilinear")
        u2 = self.up2(torch.cat([u1_up, x1], dim=1), t_emb)
        return self.out(u2)

In [ ]:
class RectifiedFlow:
    @torch.no_grad()
    def sample(self, model, cond, steps=50):
        model.eval()
        dt = 1.0 / steps
        # Start from pure noise (t=0)
        x = torch.randn_like(cond).to(DEVICE)

        for i in range(steps):
            t = torch.ones(cond.shape[0], device=DEVICE) * (i / steps)
            v = model(x, t, cond)
            x = x + v * dt
        return x

In [ ]:
model = FlowUNet().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR)
flow = RectifiedFlow()

# Training

In [ ]:
for epoch in range(EPOCHS):
    pbar = tqdm(loader)
    model.train()
    for wm, clean, _ in pbar:
        wm, clean = wm.to(DEVICE), clean.to(DEVICE)

        # Sample t in [0, 1]
        t = torch.rand(wm.shape[0], device=DEVICE)
        t_expand = t[:, None, None, None]

        x0 = torch.randn_like(clean)
        x1 = clean

        # Linear interpolation
        xt = (1 - t_expand) * x0 + t_expand * x1
        target = x1 - x0  # Velocity

        pred = model(xt, t, wm)
        loss = nn.MSELoss()(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_description(f"Epoch {epoch} Loss {loss.item():.4f}")

# Submission

In [ ]:
test_ds = BirdDataset(None, os.path.join(OUT_DIR, "test"), transform)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
for wm, _, names in tqdm(test_loader):
    samples = flow.sample(model, wm.to(DEVICE))
    for i, name in enumerate(names):
        img = transforms.ToPILImage()((samples[i].cpu().clamp(-1, 1) + 1) / 2)
        img.save(os.path.join(SUBMISSION_DIR, name))